
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Explore Your Declarative Pipeline Results</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Query the tables created by your SDP pipeline and compare them to the tables you built manually.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

*Prerequisite: You must complete **Lesson 16** (create and run the ETL Pipeline) before starting this practice. The tables explored here are created by that pipeline.*

In [0]:
# Check that the SDP pipeline tables exist
_sdp_tables = [
    "current_employees_bronze_sdp",
    "current_employees_silver_sdp",
    "total_roles_gold_sdp"
]
_missing = []

for _t in _sdp_tables:
    try:
        spark.sql(f"DESCRIBE TABLE {_t}")
    except Exception as _e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(_e):
            _missing.append(_t)
        else:
            raise _e

if _missing:
    print("The following SDP tables were not found:")
    for _t in _missing:
        print(f"  - {_t}")
    print("\nThese tables are created by running the ETL Pipeline in Lesson 16.")
    print("Go back to Lesson 16, follow the steps to create and start the pipeline,")
    print("and return here once the pipeline shows 'Completed'.")
else:
    print("All 3 SDP tables found — ready to go!")

---
### Task 1: Query the SDP Bronze table

Start by looking at the raw data that the streaming table ingested from the CSV files.

In [0]:
%sql
-- TODO: Query the SDP Bronze table
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>SELECT * FROM current_employees_bronze_sdp;</code></pre>

</details>

You should see **6 rows** — all employees from both `employees.csv` and `employees2.csv`. The streaming table ingested all CSV files from the volume in one pass, just like COPY INTO did in Lesson 12.

---
### Task 2: Query the SDP Silver table

Look at the cleaned and enriched data. Pay attention to the columns that were added by the transformation.

In [0]:
%sql
-- TODO: Query the SDP Silver table
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>SELECT * FROM current_employees_silver_sdp;</code></pre>

</details>

Notice the `Role` column is uppercased and there are `processed_timestamp` and `processed_date` columns — the same transformations from Lesson 12, but defined declaratively.

---
### Task 3: Query the SDP Gold table

Check the aggregated results.

In [0]:
%sql
-- TODO: Query the SDP Gold table
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>SELECT * FROM total_roles_gold_sdp;</code></pre>

</details>

You should see the same role counts as `total_roles_gold` from Lesson 12 — the pipeline produced the same business-ready summary, just with less code and no manual orchestration.

---
### Task 4: Compare SDP tables to your manual tables

Let's see how the results compare. Run the query below to put the manual Gold table and the SDP Gold table side by side.

In [0]:
%sql
SELECT 'Manual (Lesson 12)' AS source, Role, TotalEmployees
FROM total_roles_gold
UNION ALL
SELECT 'SDP (Lesson 16)', Role, TotalEmployees
FROM total_roles_gold_sdp
ORDER BY Role, source;

The data should match: both pipelines produced the same results from the same source files. The difference is *how* they got there: one was step-by-step imperative code across multiple notebooks, the other was three declarative SQL statements in a single notebook.

---
### Task 5: Explore data quality expectations

In Lesson 16, you added two `CONSTRAINT` expectations to the Silver layer. Let's check if they were evaluated.

1. Open **Catalog Explorer** from the left sidebar
2. Navigate to **labuser** → **get_started_de** → **current_employees_silver_sdp**
3. Click the **Quality** tab

You should see the two expectations listed:
- `valid_id` — EXPECT (ID IS NOT NULL)
- `valid_name` — EXPECT (FirstName IS NOT NULL)

Both should show 100% pass rate since the employee data has no null values.

You can also check the expectations from SQL. Run the query below to confirm there are no null IDs or names in the Silver table.

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(ID) AS non_null_ids,
  COUNT(FirstName) AS non_null_names,
  COUNT(*) - COUNT(ID) AS null_ids,
  COUNT(*) - COUNT(FirstName) AS null_names
FROM current_employees_silver_sdp;

All rows should have non-null IDs and names, meaning both expectations passed. In a production pipeline with messier data, these expectations would help you catch quality issues early.

---
### Task 6: Explore lineage

One of the advantages of SDP is that lineage is tracked automatically.

1. In **Catalog Explorer**, navigate to **total_roles_gold_sdp**
2. Click the **Lineage** tab
3. You should see the full chain: CSV files → Bronze → Silver → Gold

Compare this to the lineage for `total_roles_gold` (your manual Gold table from Lesson 12). Both show a similar chain, but the SDP lineage was established automatically from your SQL definitions. You didn't have to do anything extra.


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">Nice work! You just:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Queried all three layers of a declaratively-built pipeline</li>
      <li>Compared SDP results to your manual tables and confirmed they match</li>
      <li>Explored data quality expectations in Catalog Explorer</li>
      <li>Traced lineage from source files through Bronze, Silver, and Gold</li>
    </ul>
  </div>
</div>
</div>


<!-- CHECKPOINT: Bonus lesson complete -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 24px 28px; text-align: center;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Bonus Complete</div>
  <div style="font-size: 20pt; font-weight: 700;">Declarative Pipelines — Done!</div>
</div>

<div style="margin-top: 16px; padding: 20px 24px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.7;">
    <p>In this bonus lesson, you rebuilt the same Bronze → Silver → Gold pipeline using a completely different approach:</p>
    <ul style="padding-left: 20px; margin: 8px 0;">
      <li>Defined the full pipeline in <strong>3 SQL statements</strong> instead of multiple notebooks and manual steps</li>
      <li>Used <strong>streaming tables</strong> for incremental ingestion and <strong>materialized views</strong> for transformations</li>
      <li>Added <strong>data quality expectations</strong> that are tracked automatically</li>
      <li>Let SDP handle orchestration, compute, and execution order</li>
    </ul>
    <p>In production, most Databricks data engineers use Spark Declarative Pipelines for exactly this reason: less code, automatic orchestration, and built-in data quality.</p>
  </div>
</div>

<div style="margin-top: 16px; padding: 16px 20px; background: rgba(0,169,114,0.08); border-left: 4px solid #00A972; border-radius: 6px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Ready to clean up?</strong> Use the <strong>Reset or Clean Up Course Resources</strong> notebook to remove all tables and assets created during this course.
  </div>
</div>

</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>